# __Homework 4:__ Practical analysis with BioPython

For the homework, you are going to extend the code from the analysis of our FASTQ file in lectures 8 and 9.
Recall that the FASTQ file contains reads from a real sequencing run of influenza virus HA and NA genes.

---
The __actual sequences__ are as follows:

    5'-[end of HA]-AGGCGGCCGC-[16 X N barcode]-3'
or 

    5'-[end of NA]-AGGCGGCCGC-[16 X N barcode]-3'
---


__The end of NA is__ `...CACGATAGATAAATAATAGTGCACCAT`
    
__The end of HA is__ `...CCGGATTTGCATATAATGATGCACCAT`

---    

    
The __sequencing reads__ from the reverse end of the molecules (in 5'>3' orientation), so the sequencing reads are as follows:

    5'-[reverse complement of 16 X N barcode]-GCGGCCGCCT-[reverse complement of the end of HA]-3'
or

    5'-[reverse complement of 16 X N barcode]-GCGGCCGCCT-[reverse complement of the end of NA]-3'

---   
    
The reads can originate from **either** HA or NA, and that will be distinguished by the most 3' end of the read.
But in our example exercise in class, we did not distinguish among reads matching to HA and NA, as we didn't even look far enough into the read to tell the identity.

For the homework, your goal is to write code that extends the material from lectures 8 and 9 to also distinguish between HA and NA.
This homework can be completed almost entirely by re-using code from lecture 9. You will need to set up your analysis to do the following:
 1. Get the reverse complement of each read.
 2. Determine if it matches the expected pattern for HA and NA, and if so which one.
 3. If it matches, extract the barcode and add it to a dictionary to keep track of counts.
 4. Determine the number and distribution of barcodes for HA and NA separately.

Please include code to address each of the following questions. Please include code comments to explain what your code is attempting to accomplish. Don't forget to include references to the sources you used to obtain your answer, including your classmates (if you are working in groups).  

1. How many reads map to HA, and how many reads map to NA?

In [161]:
# your code here...
import re
import Bio.SeqIO
import Bio.Seq

reads = Bio.SeqIO.parse('barcodes_R1.fastq', format='fastq') #I'm using Bio.seqI0 to import the fastq file
seqreads = list(reads) #convert the SeqIO object to a list

HA_readcount = 0 #initializing HA read count
NA_readcount = 0 #initializing NA read count
HA_read = [] #initializing list to hold HA reads
NA_read = [] #initializing list to hold NA reads

seqreads_Seq = []
for seqrecord in seqreads:
    sequence = seqrecord.seq # isolate the sequence from the seqrecord
    seqreads_Seq.append(sequence) # add string sequence to list

HA_pat = re.compile('CCGGATTTGCATATAATGATGCACCAT(?P<AGGCGGCCGC>)') #establishing HA pattern
NA_pat = re.compile('CACGATAGATAAATAATAGTGCACCAT(?P<AGGCGGCCGC>)') #establishing NA pattern


for seq in seqreads_Seq:
    seq_string = str(seq) #convert Seq object to string
    seq_revcomp = str(Bio.Seq.reverse_complement(seq)) #get reverse complement of sequence
    HA_match = HA_pat.search(seq_revcomp) #search for HA pattern match
    NA_match = NA_pat.search(seq_revcomp) #search for NA pattern match
    if HA_match:
        HA_readcount += 1 #add to read count if NA pattern found
        HA_read.append(seq_revcomp) # add it to HA read list 

    elif NA_match:
        NA_readcount += 1 #add to read count if HA pattern found
        NA_read.append(seq_revcomp)# add it to NA read list

print("Number of HA reads:", HA_readcount)
print("Number of NA reads:", NA_readcount)

Number of HA reads: 5409
Number of NA reads: 4122


2. How many HA sequences did not have a valid barcode? Also anwer the same question for NA.

In [162]:
# your code here...
# call it once for HA and once for NA
# pass through 16 as bclen when calling 

def read_barcode(virus_match, bclen, upstream): #defining function to read barcodes
    barcode_counts = {} #initializing barcode counts dictionary
    invalid_count = 0 #making invalid NA count 0
    pat = re.compile(upstream+"(?P<barcode>[ATGC]{"+str(bclen)+"})") #establishing pattern to find barcodes
    for seq in virus_match: #looping through the HA or NA
        match = pat.search(seq) #searching for pattern match
        if match is None: 
            invalid_count += 1 #add to invalid count if no match found
        else:
            barcode = match.group('barcode') #  isolate barcode from match
            if barcode in barcode_counts: #if barcode already in dictionary
                barcode_counts[barcode] += 1
            else: #if barcode not in dictionary
                barcode_counts[barcode] = 1 

    print("invalid barcode counts:", invalid_count)

    m=0 #initializing max counter
    max_barcode = "" #empty for most frequent barcode to be put in later
    for barcode, count in barcode_counts.items():
        if count > m: #if current count is greater than max count
            m = count #update max count
            max_barcode = barcode #update most frequent barcode
    print("most frequent barcode:", max_barcode, "with count:", m)


In [163]:
read_barcode(HA_read, 16, upstream='AGGCGGCCGC')
#for i in HA_read:
 #   print(i)
read_barcode(NA_read, 16, upstream='AGGCGGCCGC')



invalid barcode counts: 160
most frequent barcode: CCCGACCCGACATTAA with count: 155
invalid barcode counts: 213
most frequent barcode: ACCAGTTCTCCCCGGG with count: 152


3. What is the HA barcode with the most counts (and how many counts)? Also answer the same question for NA.

    _Hint: you will need to find the key associated with the maximum value in your dictionary. There are many ways to do this._

In [164]:
# your code here...
#code is in the function above. 
#most frequent HA barcode: CCCGACCCGACATTAA with count of 155
#most frequent barcode: ACCAGTTCTCCCCGGG with count: 152